In [3]:
# =============================================================================
# MODEL 01 - RIDGE & LASSO REGRESSION
# Task: Predict Crime_Count based on Location and Date
# =============================================================================

import warnings
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG
# =============================================================================
DATA_PATH = "./crime_per_capita_df.csv"
TARGET = "Crime_Count"

TRAIN_YEARS = [2020, 2021, 2022, 2023]
TEST_YEARS = [2024]

LASSO_ALPHAS = [0.01, 0.1, 1.0, 10.0]
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]

HOTSPOT_PERCENTS = [5, 10, 20]
REPORT_TXT_PATH = "evaluation_results_model_01.txt"

FEATURES = [
    "LSOA_Latitude",
    "LSOA_Longitude",
    "LSOA_Shape_Area",
    "LSOA_Shape_Length",
    "Year",
    "Month",
    "month_sin",
    "month_cos",
    "is_summer",
    "is_winter",
    "is_spring",
    "is_autumn",
    "Total_Population",
    "pop_density",
    "income_per_capita",
    "Income_Deprivation",
    "Max_Temperature_Celsius",
    "Min_Temperature_Celsius",
    "Rainfall_mm",
    "Sunshine_Hours",
    "Air_Frost_Days",
    "crime_lag_1m",
    "crime_lag_2m",
    "crime_lag_3m",
    "crime_lag_6m",
    "crime_rolling_3m_mean",
    "crime_rolling_3m_std",
    "crime_rolling_6m_mean",
    "crime_rolling_6m_std",
    "msoa_avg_crime",
]


# =============================================================================
# 2. HELPERS
# =============================================================================
def safe_divide(numerator, denominator):
    denominator = float(denominator)
    if denominator == 0.0:
        return np.nan
    return float(numerator) / denominator


def round_or_nan(value, digits=4):
    if pd.isna(value) or not np.isfinite(value):
        return np.nan
    return round(float(value), digits)


def fmt4(value):
    if pd.isna(value) or not np.isfinite(value):
        return "nan"
    return f"{float(value):.4f}"


def evaluate(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-9))) * 100

    print(f"\n{'=' * 45}")
    print(f"  {model_name}")
    print(f"{'=' * 45}")
    print(f"  MAE   : {mae:.4f}   (avg error in crime count units)")
    print(f"  RMSE  : {rmse:.4f}  (penalises large errors more)")
    print(f"  R2    : {r2:.4f}   (1.0 = perfect, 0 = baseline mean)")
    print(f"  MAPE  : {mape:.2f}%  (mean absolute % error)")

    return {
        "Model": model_name,
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "R2": round(r2, 4),
        "MAPE": round(mape, 2),
    }


# =============================================================================
# CRIME HOTSPOT EVALUATION METRICS
# PAI — Prediction Accuracy Index
# RRI — Recapture Rate Index
# PEI — Prediction Efficiency Index
# =============================================================================
def crime_hotspot_metrics(y_true, y_pred, hotspot_percent=10, model_name="Model"):
    if not (0 < hotspot_percent <= 100):
        raise ValueError("hotspot_percent must be in (0, 100].")

    df_eval = pd.DataFrame(
        {
            "actual": np.asarray(y_true, dtype=float).reshape(-1),
            "predicted": np.asarray(y_pred, dtype=float).reshape(-1),
        }
    ).dropna().reset_index(drop=True)

    if df_eval.empty:
        return {
            "Model": model_name,
            "Hotspot_%": hotspot_percent,
            "Crimes_Captured": 0,
            "Total_Crimes": 0,
            "RRI": np.nan,
            "PAI": np.nan,
            "PEI": np.nan,
        }

    threshold = np.percentile(df_eval["predicted"], 100 - hotspot_percent)
    df_eval["hotspot"] = df_eval["predicted"] >= threshold

    crimes_in_hotspots = float(df_eval.loc[df_eval["hotspot"], "actual"].sum())
    total_crimes = float(df_eval["actual"].sum())
    hotspot_area = int(df_eval["hotspot"].sum())
    total_area = int(len(df_eval))

    rri = safe_divide(crimes_in_hotspots, total_crimes)
    area_ratio = safe_divide(hotspot_area, total_area)
    pai = safe_divide(rri, area_ratio) if pd.notna(rri) and pd.notna(area_ratio) else np.nan
    pei = pai  # simplified PEI approximation

    print("\nCrime Hotspot Evaluation")
    print("-" * 40)
    print(f"Model             : {model_name}")
    print(f"Hotspot area used : top {hotspot_percent}%")
    print(f"Crimes captured   : {crimes_in_hotspots:.0f} / {total_crimes:.0f}")
    print(f"RRI               : {fmt4(rri)}")
    print(f"PAI               : {fmt4(pai)}")
    print(f"PEI               : {fmt4(pei)}")

    return {
        "Model": model_name,
        "Hotspot_%": hotspot_percent,
        "Crimes_Captured": int(round(crimes_in_hotspots)),
        "Total_Crimes": int(round(total_crimes)),
        "RRI": round_or_nan(rri, 4),
        "PAI": round_or_nan(pai, 4),
        "PEI": round_or_nan(pei, 4),
    }


def write_evaluation_report(output_path, regression_df, hotspot_df):
    reg_cols = ["Model", "MAE", "RMSE", "R2", "MAPE"]
    hs_cols = ["Model", "Hotspot_%", "Crimes_Captured", "Total_Crimes", "RRI", "PAI", "PEI"]

    regression_out = regression_df[reg_cols].copy()
    hotspot_out = hotspot_df[hs_cols].copy().sort_values(["Model", "Hotspot_%"])

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("# =============================================================================\n")
        f.write("# CRIME HOTSPOT EVALUATION METRICS\n")
        f.write("# PAI — Prediction Accuracy Index\n")
        f.write("# RRI — Recapture Rate Index\n")
        f.write("# PEI — Prediction Efficiency Index\n")
        f.write("# =============================================================================\n\n")
        f.write("Definitions\n")
        f.write("RRI = Crimes captured in hotspots / Total actual crimes\n")
        f.write("PAI = RRI / Hotspot area ratio\n")
        f.write("PEI = RRI / Hotspot area ratio (simplified approximation)\n\n")
        f.write("# =============================================================================\n")
        f.write("# REGRESSION EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(regression_out.to_string(index=False))
        f.write("\n\n")
        f.write("# =============================================================================\n")
        f.write("# HOTSPOT EVALUATION SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(hotspot_out.to_string(index=False))
        f.write("\n")


# =============================================================================
# 3. LOAD DATA
# =============================================================================
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(
    f"Date range: {df['Year'].min()}-{df['Month'].min():02d} to "
    f"{df['Year'].max()}-{df['Month'].max():02d}"
)
print(f"Unique LSOAs: {df['LSOA_Code'].nunique()}")


# =============================================================================
# 4. FEATURE VALIDATION
# =============================================================================
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"\nWARNING - missing columns: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]

print(f"\nUsing {len(FEATURES)} features to predict '{TARGET}'")


# =============================================================================
# 5. TRAIN / TEST SPLIT (TIME-BASED)
# =============================================================================
train = df[df["Year"].isin(TRAIN_YEARS)].copy()
test = df[df["Year"].isin(TEST_YEARS)].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test = test[FEATURES]
y_test = test[TARGET]

print(f"\nTrain size : {len(X_train):,} rows ({TRAIN_YEARS[0]}-{TRAIN_YEARS[-1]})")
print(f"Test size  : {len(X_test):,} rows ({TEST_YEARS[0]})")


# =============================================================================
# 6. SCALING
# =============================================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# =============================================================================
# 7. LASSO REGRESSION
# =============================================================================
print("\n--- Fitting Lasso Regression ---")
lasso_results = []

for alpha in LASSO_ALPHAS:
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_scaled, y_train)
    preds = np.clip(lasso.predict(X_test_scaled), 0, None)

    res = evaluate(f"Lasso (alpha={alpha})", y_test, preds)
    n_zero = int(np.sum(lasso.coef_ == 0))
    print(f"  Features zeroed out: {n_zero} / {len(FEATURES)}")

    lasso_results.append({**res, "alpha": alpha, "model_obj": lasso, "preds": preds})

best_lasso = min(lasso_results, key=lambda x: x["MAE"])
best_lasso_model = best_lasso["model_obj"]
best_lasso_preds = best_lasso["preds"]
print(f"\nBest Lasso alpha: {best_lasso['alpha']} (MAE={best_lasso['MAE']})")


# =============================================================================
# 8. RIDGE REGRESSION
# =============================================================================
print("\n--- Fitting Ridge Regression ---")
ridge_results = []

for alpha in RIDGE_ALPHAS:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    preds = np.clip(ridge.predict(X_test_scaled), 0, None)

    res = evaluate(f"Ridge (alpha={alpha})", y_test, preds)
    ridge_results.append({**res, "alpha": alpha, "model_obj": ridge, "preds": preds})

best_ridge = min(ridge_results, key=lambda x: x["MAE"])
best_ridge_model = best_ridge["model_obj"]
best_ridge_preds = best_ridge["preds"]
print(f"\nBest Ridge alpha: {best_ridge['alpha']} (MAE={best_ridge['MAE']})")


# =============================================================================
# 9. FEATURE INSPECTION
# =============================================================================
print("\n--- Top 10 Most Important Features (Best Ridge) ---")
coef_df = pd.DataFrame(
    {
        "Feature": FEATURES,
        "Coefficient": best_ridge_model.coef_,
        "Abs_Coef": np.abs(best_ridge_model.coef_),
    }
).sort_values("Abs_Coef", ascending=False)
print(coef_df[["Feature", "Coefficient"]].head(10).to_string(index=False))

print("\n--- Features Kept by Best Lasso (non-zero coefficients) ---")
lasso_coef_df = pd.DataFrame({"Feature": FEATURES, "Coefficient": best_lasso_model.coef_})
kept = lasso_coef_df[lasso_coef_df["Coefficient"] != 0].sort_values(
    "Coefficient", key=abs, ascending=False
)
dropped = lasso_coef_df[lasso_coef_df["Coefficient"] == 0]
print(f"Kept   : {len(kept)} features")
print(f"Dropped: {len(dropped)} features")
print(kept[["Feature", "Coefficient"]].to_string(index=False))


# =============================================================================
# 10. COMPARISON TABLE + CSV EXPORT
# =============================================================================
print("\n--- FINAL COMPARISON: Best Ridge vs Best Lasso ---")
summary = pd.DataFrame(
    [
        {k: v for k, v in best_ridge.items() if k not in ["model_obj", "preds"]},
        {k: v for k, v in best_lasso.items() if k not in ["model_obj", "preds"]},
    ]
)
print(summary[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_string(index=False))

summary[["Model", "MAE", "RMSE", "R2", "MAPE"]].to_csv("model_comparison.csv", index=False)
print("\nResults saved to model_comparison.csv")


# =============================================================================
# 11. SAMPLE PREDICTIONS
# =============================================================================
print("\n--- Sample Predictions vs Actual (Best Lasso, first 10 test rows) ---")
sample_lasso = test[["LSOA_Code", "Year", "Month", TARGET]].copy().head(10)
sample_lasso["Predicted"] = np.round(best_lasso_preds[:10], 1)
sample_lasso["Error"] = np.round(np.abs(sample_lasso[TARGET] - sample_lasso["Predicted"]), 1)
print(sample_lasso.to_string(index=False))

print("\n--- Sample Predictions vs Actual (Best Ridge, first 10 test rows) ---")
sample_ridge = test[["LSOA_Code", "Year", "Month", TARGET]].copy().head(10)
sample_ridge["Predicted"] = np.round(best_ridge_preds[:10], 1)
sample_ridge["Error"] = np.round(np.abs(sample_ridge[TARGET] - sample_ridge["Predicted"]), 1)
print(sample_ridge.to_string(index=False))


# =============================================================================
# 12. CRIME HOTSPOT EVALUATION
# =============================================================================
print("\n--- Crime Forecasting Indices ---")
hotspot_records = []

for model_name, model_preds in [
    ("Best Lasso", best_lasso_preds),
    ("Best Ridge", best_ridge_preds),
]:
    for pct in HOTSPOT_PERCENTS:
        result = crime_hotspot_metrics(
            y_true=y_test.values,
            y_pred=model_preds,
            hotspot_percent=pct,
            model_name=model_name,
        )
        hotspot_records.append(result)

hotspot_df = pd.DataFrame(hotspot_records)
print("\nHotspot Evaluation Summary")
print(hotspot_df.to_string(index=False))


# =============================================================================
# 13. SAVE EVALUATION RESULTS TO A SINGLE TEXT FILE
# =============================================================================
write_evaluation_report(REPORT_TXT_PATH, summary, hotspot_df)
print(f"\nEvaluation report saved to {REPORT_TXT_PATH}")

# =============================================================================
# 14. EXPORT BEST MODEL
# =============================================================================

# Select best model using MAE (lower is better)
candidate_models = [best_lasso, best_ridge]
best_overall = min(candidate_models, key=lambda x: x["MAE"])

# Bundle everything needed for future inference
best_model_package = {
    "model_name": best_overall["Model"],
    "model": best_overall["model_obj"],
    "scaler": scaler,
    "features": FEATURES,
    "target": TARGET,
    "metric_used": "MAE",
    "metrics": {
        "MAE": best_overall["MAE"],
        "RMSE": best_overall["RMSE"],
        "R2": best_overall["R2"],
        "MAPE": best_overall["MAPE"],
    },
    "train_years": TRAIN_YEARS,
    "test_years": TEST_YEARS,
}

artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)

artifact_path = artifact_dir / "best_model_model_01.joblib"
joblib.dump(best_model_package, artifact_path)

print("\nBest model export complete")
print(f"Model selected : {best_overall['Model']}")
print(f"Saved artifact : {artifact_path}")

# Optional: append export summary to your existing text report
if "REPORT_TXT_PATH" in globals():
    with open(REPORT_TXT_PATH, "a", encoding="utf-8") as f:
        f.write("\n\n# =============================================================================\n")
        f.write("# BEST MODEL EXPORT SUMMARY\n")
        f.write("# =============================================================================\n")
        f.write(f"Model selected : {best_overall['Model']}\n")
        f.write(f"Metric used    : MAE\n")
        f.write(f"MAE            : {best_overall['MAE']}\n")
        f.write(f"RMSE           : {best_overall['RMSE']}\n")
        f.write(f"R2             : {best_overall['R2']}\n")
        f.write(f"MAPE           : {best_overall['MAPE']}\n")
        f.write(f"Artifact path  : {artifact_path}\n")

Dataset shape: (304336, 39)
Date range: 2020-01 to 2024-12
Unique LSOAs: 12415

Using 30 features to predict 'Crime_Count'

Train size : 241,710 rows (2020-2023)
Test size  : 62,626 rows (2024)

--- Fitting Lasso Regression ---

  Lasso (alpha=0.01)
  MAE   : 4.8124   (avg error in crime count units)
  RMSE  : 8.9729  (penalises large errors more)
  R2    : 0.9382   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 44.25%  (mean absolute % error)
  Features zeroed out: 0 / 30

  Lasso (alpha=0.1)
  MAE   : 4.6965   (avg error in crime count units)
  RMSE  : 8.7972  (penalises large errors more)
  R2    : 0.9406   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 43.17%  (mean absolute % error)
  Features zeroed out: 15 / 30

  Lasso (alpha=1.0)
  MAE   : 4.6464   (avg error in crime count units)
  RMSE  : 8.7064  (penalises large errors more)
  R2    : 0.9418   (1.0 = perfect, 0 = baseline mean)
  MAPE  : 47.44%  (mean absolute % error)
  Features zeroed out: 24 / 30

  Lasso (alpha=10.0)
  MAE 